# کارگر 2 از 4: تبدیل PDFها (نسخه‌ی 13)

این نوت‌بوک فقط PDFها را به متن ساختاریافته تبدیل می‌کند (کُندترین مرحله) و نتیجه را در
`RAG-Optics/_RAG_processed` در Drive می‌گذارد. هر کارگر سهم جداگانه‌ای از PDFها را دارد، پس
کارگرهای ۱ تا 4 را **هم‌زمان در تب‌های جدا** باز و اجرا کنید. مدل زبانی و پایگاه داده لازم نیست.

- کارت گرافیک اختیاری است (*Runtime → Change runtime type → T4 GPU* سریع‌تر است؛ اگر کولب
  کارت گرافیک نداد، روی CPU هم کار می‌کند، فقط کُندتر).
- تب را باز نگه دارید. اگر قطع شد، دوباره اجرا کنید: مقاله‌های تمام‌شده دوباره تبدیل نمی‌شوند.
- اگر کولب اجازه‌ی باز کردن این همه جلسه را نداد، هر تعداد که شد کافی است: سلول ۵ نوت‌بوک اصلی
  باقی‌مانده را انجام می‌دهد.
- بعد از تمام شدن همه، در نوت‌بوک اصلی سلول‌های ۱ تا ۵ را اجرا کنید.

In [ ]:
#@title ▶️ اجرای کارگر 2
WORKER = 2  #@param {type:"integer"}
WORKERS = 4  #@param {type:"integer"}
COLLECTION = "/content/drive/MyDrive/RAG-Optics"  #@param {type:"string"}

import os
# Keep Colab's preinstalled JAX/TensorFlow from reserving the GPU memory
# (otherwise the embedding model and Ollama run out of memory).
os.environ.update({'JAX_PLATFORMS': 'cpu', 'XLA_PYTHON_CLIENT_PREALLOCATE': 'false',
                   'TF_FORCE_GPU_ALLOW_GROWTH': 'true', 'USE_TF': '0', 'USE_FLAX': '0',
                   'TF_CPP_MIN_LOG_LEVEL': '3'})
from google.colab import drive
drive.mount('/content/drive')

BRANCH = "claude/festive-bardeen-nnw123"  #@param {type:"string"}
# Leave the project folder before deleting it (re-running this cell used to fail
# with "Unable to read current working directory").
%cd /content
!rm -rf /content/RAG && git clone -q --depth 1 -b {BRANCH} https://github.com/aminsadidi/RAG.git /content/RAG
%cd /content/RAG
!pip install -q -e . 2>&1 | grep -iE "^error" || true
# An editable install is only seen by Python after a restart; adding the source
# folder to the path makes it importable in this session right away.
import sys
if '/content/RAG/src' not in sys.path:
    sys.path.insert(0, '/content/RAG/src')

os.environ['MATRAG_DATA_DIR'] = '/content/worker_data'  # only the link to the collection lives here
os.environ['MATRAG_CORPUS'] = 'rag-optics'
!matrag link "{COLLECTION}" > /dev/null
!matrag convert --shard {WORKER - 1} --shards {WORKERS} 2>&1 | grep -vE "Warning|it/s\]"
print(f'✅ Worker {WORKER} finished.')